# Chapter 1: Basic Prompt Structure

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `get_completion` helper function.

In [ ]:
# !pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        extra_body={"temperature": 0.0},  # anthropic SDK 1.x dropped temperature from the typed
                                           # create() signature; extra_body still sends it on the wire
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Lesson

Anthropic offers two APIs:

- **[Messages API](https://docs.anthropic.com/en/api/messages)** *(current)* — the standard API for all new projects; used throughout this tutorial
- **[Text Completions API](https://docs.anthropic.com/en/api/complete)** *(legacy / deprecated)* — older format; documented below for reference

---

### Messages API

At minimum, a call to Claude using the Messages API requires the following parameters:

- `model`: the [API model name](https://docs.anthropic.com/en/docs/about-claude/models) of the model you intend to call

- `max_tokens`: the maximum number of tokens to generate before stopping. This is a *hard* stop — Claude may be cut off mid-word or mid-sentence. When using extended thinking, thinking tokens count toward this limit.

- `messages`: an array of input messages. Models operate on alternating `user` and `assistant` conversational turns.
  - Each message must be an object with a `role` (`"user"` or `"assistant"`) and `content` (string or a list of content blocks)
  - Messages **must alternate** between roles, and the first message **must** use the `"user"` role

There are also optional parameters:

- `system`: a system prompt — context, instructions, or guidelines provided to Claude *before* the conversation messages

- `temperature` *(float, 0–1, default 1)*: controls response variability. Lower values yield more deterministic outputs. This tutorial uses `temperature=0`.
  > **SDK note:** `anthropic` Python SDK 1.x removed `temperature`/`top_p`/`top_k` from the typed `messages.create()` signature (newer models like Opus 5 and Sonnet 5 reject them outright — thinking replaces the need for a sampling knob). For models that still accept it, like Haiku 4.5, pass it via `extra_body={"temperature": 0.0}` instead of as a direct keyword argument.

- `top_p` *(float, 0–1)*: nucleus sampling — restricts token selection to the smallest set whose cumulative probability ≥ `top_p`. Use *either* `temperature` or `top_p`, not both.

- `top_k` *(int)*: limits token selection to the top-K most probable tokens

- `stop_sequences` *(list of strings)*: Claude stops generating as soon as it produces any of these strings

- `stream` *(bool)*: if `True`, incrementally streams the response via server-sent events instead of waiting for full completion

- `tools` *(list)*: a list of tool/function definitions the model may call (name, description, input schema)

- `tool_choice`: how Claude selects tools — `{"type": "auto"}`, `{"type": "any"}`, or `{"type": "tool", "name": "..."}` to force a specific tool

- `thinking`: enables extended (chain-of-thought) reasoning before answering.  
  `{"type": "enabled", "budget_tokens": 5000}` — `budget_tokens` must be ≥ 1,024 and less than `max_tokens`.  
  Use `"type": "adaptive"` on newer models to let Claude decide when thinking is needed.

- `metadata`: an object for passing extra information, e.g. `{"user_id": "user_123"}`

For the complete parameter reference, visit the [Messages API docs](https://docs.anthropic.com/en/api/messages).

---

### Text Completions API *(Legacy — for reference only)*

> **Note:** The Text Completions API is deprecated and not supported in `anthropic` SDK v1+. All new code should use the Messages API above.

Unlike the Messages API's structured messages list, the Text Completions API uses a single `prompt` string that manually encodes the conversation using a Human/Assistant format.

**Required parameters:**

- `model`: the API model name

- `prompt`: a string that must follow the pattern `\n\nHuman: {text}\n\nAssistant:`. Claude continues generating from the `Assistant:` turn. System instructions must be prepended manually to this string.

- `max_tokens_to_sample`: the maximum number of tokens to generate

**Optional parameters:**

- `temperature` *(0–1)*: controls randomness
- `top_p`, `top_k`: sampling controls (same semantics as Messages API)
- `stop_sequences`: list of strings to stop generation
- `stream`: enable streaming

**Example prompt format:**
```
\n\nHuman: What is the capital of France?\n\nAssistant:
```

**Why it was replaced:** The fixed string format makes it hard to manage multi-turn conversations, inject system prompts cleanly, or attach structured content like images. The Messages API solves all of this with explicit `role`/`content` objects.


### Examples

Let's take a look at how Claude responds to some correctly-formatted prompts. For each of the following cells, run the cell (`shift+enter`), and Claude's response will appear below the block.

In [23]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

Hey! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need—whether that's answering questions, working through problems, brainstorming ideas, or just having a conversation. What's on your mind today?


In [24]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

The ocean appears **blue** to most people, though the shade varies depending on several factors:

- **Sky reflection**: The water reflects the sky, so it looks bluer on clear days
- **Depth**: Deeper water appears darker blue; shallow water may look turquoise or green
- **Lighting**: The angle of sunlight affects the color you see
- **Sediment and algae**: These can make water look green, brown, or other colors
- **Location**: Different seas and oceans have different typical colors

In reality, water itself is slightly blue because it absorbs red wavelengths of light better than blue wavelengths, but this color is faint in small amounts. The ocean's blue is largely due to these reflected and light-absorption factors combined.


In [25]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Celine Dion was born in 1968.


Now let's take a look at some prompts that do not include the correct Messages API formatting. For these malformatted prompts, the Messages API returns an error.

First, we have an example of a Messages API call that lacks `role` and `content` fields in the `messages` array.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
      #  extra_body={"temperature": 0.0},
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Here's a prompt that fails to alternate between the `user` and `assistant` roles.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
      #  extra_body={"temperature": 0.0},
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

`user` and `assistant` messages **MUST alternate**, and messages **MUST start with a `user` turn**. You can have multiple `user` & `assistant` pairs in a prompt (as if simulating a multi-turn conversation). You can also put words into a terminal `assistant` message for Claude to continue from where you left off (more on that in later chapters).

#### System Prompts

You can also use **system prompts**. A system prompt is a way to **provide context, instructions, and guidelines to Claude** before presenting it with a question or task in the "User" turn. 

Structurally, system prompts exist separately from the list of `user` & `assistant` messages, and thus belong in a separate `system` parameter (take a look at the structure of the `get_completion` helper function in the [Setup](#setup) section of the notebook). 

Within this tutorial, wherever we might utilize a system prompt, we have provided you a `system` field in your completions function. Should you not want to use a system prompt, simply set the `SYSTEM_PROMPT` variable to an empty string.

#### System Prompt Example

In [32]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Great question! Rather than jumping to the answer, let me pose some questions to help you think through this:

1. Have you noticed that the sky appears different colors at different times of day—like orange or red at sunset? What do you think causes that change?

2. What do you know about sunlight? Is it actually just one color, or could it be made up of multiple colors combined?

3. When light travels through our atmosphere, it interacts with tiny particles and gases. How do you think different colors of light might behave differently when hitting these tiny particles?

4. If you've ever seen a rainbow, you know sunlight can be separated into different colors. Which colors have shorter wavelengths, and which have longer wavelengths? Does that distinction seem important here?

5. Why do you think we see blue specifically, rather than other colors like red or yellow?

These questions should guide your thinking toward understanding the phenomenon!


Why use a system prompt? A **well-written system prompt can improve Claude's performance** in a variety of ways, such as increasing Claude's ability to follow rules and instructions. For more information, visit our documentation on [how to use system prompts](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) with Claude.

Now we'll dive into some exercises. If you would like to experiment with the lesson prompts without changing any content above, scroll all the way to the bottom of the lesson notebook to visit the [**Example Playground**](#example-playground).

---

## Exercises
- [Exercise 1.1 - Counting to Three](#exercise-11---counting-to-three)
- [Exercise 1.2 - System Prompt](#exercise-12---system-prompt)

### Exercise 1.1 - Counting to Three
Using proper `user` / `assistant` formatting, edit the `PROMPT` below to get Claude to **count to three.** The output will also indicate whether your solution is correct.

In [33]:
# Prompt - this is the only field you should change
PROMPT = "Count to Three"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

1
2
3

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


❓ If you want a hint, run the cell below!

In [34]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

The grading function in this exercise is looking for an answer that contains the exact Arabic numerals "1", "2", and "3".
You can often get Claude to do what you want simply by asking.


### Exercise 1.2 - System Prompt

Modify the `SYSTEM_PROMPT` to make Claude respond like it's a 3 year old child.

In [35]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "you are a 3 year old. when responding include the baby noices like coo and gaa and laughing sounds"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

*giggles and coos* 

Gaa gaa! Da sky is SO SO BIG! *laughs* Coo coo coo! 

It goes up up UP and all around and around! *babbles excitedly* Goo goo gaa! You can't see where it stops 'cause it's REALLY big! *coos and chuckles* 

Baba baba! Da sky is like... like a REALLY REALLY big blanky that goes all over everything! *giggles* Coo! And it has the sun and the moon in it! Gaa gaa gaa! *happy baby sounds*

Hehehehe! So big you can't even measure it with your tiny hands! *coos contentedly* Goo goo coo coo!

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


❓ If you want a hint, run the cell below!

In [36]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

The grading function in this exercise is looking for answers that contain "soo" or "giggles".
There are many ways to solve this, just by asking!


### Congrats!

If you've solved all exercises up until this point, you're ready to move to the next chapter. Happy prompting!

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect Claude's responses.

In [37]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

Hi! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need. How are you doing today?


In [38]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

The ocean appears **blue** to most people, though its color varies depending on conditions:

- **Deep blue** in clear, deep waters far from shore
- **Lighter blue or turquoise** in shallow, tropical areas
- **Green** in some regions where algae or sediment is present
- **Gray or dark** in stormy conditions or cloudy weather

The blue color primarily comes from water absorbing red wavelengths of light and reflecting blue wavelengths back to us. The sky's reflection also contributes to how we perceive the ocean's color.


In [39]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Celine Dion was born in 1968.


In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        extra_body={"temperature": 0.0},
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        extra_body={"temperature": 0.0},
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [42]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Great question! Rather than me explaining it, let me ask you some questions to think through this:

1. What do you already know about light and how it travels?

2. Have you noticed that the sky looks different colors at different times of day—like orange or red at sunset? What do you think causes that change?

3. If sunlight appears white or yellow to us, what would need to happen to it for the sky to look blue specifically?

4. Why do you think the sky might be blue in some places but not others (like near the horizon versus directly overhead)?

5. What role do you think the gases and particles in our atmosphere might play in this?
